# Clasificador v6 – NTP 900.058 (solo datasets públicos)
Parte de **v5** y ataca directo lo que todavía falla: **orgánicos, peligrosos (Rojo), plástico y vidrio**.

**Qué cambia:** hasta v5 solo usábamos fotos de *otros* datasets. El examen del docente usa la carpeta **test** de 3 datasets
públicos; sus fuentes originales tienen **miles de fotos más que no están en ese test**. v6 entrena con esas fotos
(mismo tipo de imagen que el examen), que es la práctica estándar: **entrenar con el resto del dataset y evaluar con su test**.

| Falla en v5 | Fuente nueva (sin las fotos del examen) | Color |
|---|---|---|
| Plástico 73%, vidrio 93% | **TrashNet** completo (cartón, papel, metal, plástico, vidrio, basura) | Azul / Amarillo / Blanco / Gris / Negro |
| Orgánicos 59% | **Waste Classification** (Kaggle techsash), clase O | Marrón |
| Peligrosos 30% | **Medical Waste 4.0** (Zenodo): gasas, guantes, gorros, lentes, cubrezapatos, tubos, bolsas de orina; RGB **y** estéreo en grises | Rojo |

**Reglas de juego limpio (para tu informe):**
- Ninguna foto del examen entra al entrenamiento. Se quitan por **3 filtros**: mismo nombre de archivo original, misma
  sesión de captura (Medical Waste 4.0: ±20 s de la foto del examen, incluye su par estéreo) y huella pHash casi igual (≤12).
- Train y val se separan por **objeto** (las fotos de una misma sesión van juntas), para que val no salga inflado.
- El modelo se **elige con val** (a color y en grises). El examen solo se mide al final.
- Ajuste fino de los 7 colores calibrado en val y grabado dentro del `.pt`.
- `clasificador.pt` solo se reemplaza si v6 **no baja más de 2 puntos en ningún color** de val frente a v5, ni en el test viejo, ni en la cámara.
- Declara estas 3 fuentes en tu informe.

**Si Colab se desconecta:** vuelve a ejecutar **todo desde la celda 1** (Entorno de ejecución → Ejecutar todo).
La partición train/val es fija entre sesiones y el entrenamiento se **salta solo** si ya está en Drive con esa misma partición.
**Entorno:** GPU T4. ~1.5 h (Medical Waste 4.0 pesa ~1-2 GB).

## 1. Instalar y verificar la GPU

In [ ]:
!pip install -q "ultralytics>=8.3,<8.4" imagehash kagglehub
import torch
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Conectar Drive y configuración

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, re, shutil, zipfile, random, json, glob, urllib.request
from collections import Counter, defaultdict
from datetime import datetime

DRIVE_DIR = "/content/drive/MyDrive/data basurita"
ZIP_V1 = f"{DRIVE_DIR}/dataset_ntp900058.zip"     # particion vieja (congelada)
MODELOS = f"{DRIVE_DIR}/modelos"
MODELO_V4 = f"{MODELOS}/clasificador_v4.pt"
MODELO_V5 = f"{MODELOS}/clasificador_v5.pt"         # v5 entregado (con ajuste de Rojo): el que hay que superar
RUNS = f"{DRIVE_DIR}/runs_clasificador"
DATASET = "/content/dataset_v6"
CAMARA = "/content/test_camara"
EXAMEN_DIR = f"{DRIVE_DIR}/examen"                  # aqui subes los 'test' del examen (opcional)
REVISION = f"{DRIVE_DIR}/revision_v6"

SEMILLA = 42
PCT_VAL = 0.15
GAP_SEG = 60
MAX_CAM_POR_COLOR = 60
MAX_PUBLICO_POR_COLOR = 800     # todas las fuentes publicas juntas, por color
PCT_GRIS = 0.40                 # parte del train que se pasa a escala de grises
ENTRENAR_S = False              # el modelo nano basta y es el que corre bien en Raspberry
# FUENTES DEL MISMO TIPO QUE EL EXAMEN (se usan SIN las fotos del examen: filtro por nombre, sesion y pHash)
USAR_TRASHNET = True            # plastico, vidrio, carton, papel, metal (mismo estilo que garbage.v1i)
USAR_WASTE_CLS_TECHSASH = True  # clase O -> Marron (mismo estilo que 'Organik')
USAR_MEDICAL_WASTE_40 = True    # -> Rojo (mismo estilo que 'Infectious')
MAX_DOMINIO_POR_COLOR = 1000    # tope de estas fuentes por color (aparte de MAX_PUBLICO_POR_COLOR)
SESION_MW_SEG = 20              # Medical Waste 4.0: se quita todo lo capturado a +-20 s de una foto del examen
# EPP medico publico (mascarillas, jeringas, pinzas) del dataset Kaggle 'sumn2u/medical-waste', SIN la parte de
# Medical Waste 4.0. Ojo: el examen tambien tiene fotos de esa fuente, por eso se filtra MAS estricto contra el
# examen (distancia <= 12). Si lo usas, declaralo en tu informe.
USAR_EPP_PUBLICO = True
USAR_BIOMEDICO_EXTRA = True     # Kaggle mario78/biomedical-waste-classification-images-dataset (jeringas, ampollas...)
USAR_FRUTAS_PODRIDAS = True     # Kaggle sriramr/fruits-fresh-and-rotten-for-classification (solo podridas -> Marron)
LADO_MAX = 512
EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
random.seed(SEMILLA)

NOMBRES = {
    "Azul": "azul_papel_carton",
    "Amarillo": "amarillo_metales",
    "Blanco": "blanco_plasticos",
    "Gris": "gris_vidrio",
    "Marron": "marron_organicos",
    "Rojo": "rojo_peligrosos",
    "Negro": "negro_generales",
}
LARGO_A_CORTO = {v: k for k, v in NOMBRES.items()}

def imagenes(carpeta):
    if not os.path.isdir(carpeta):
        return []
    return sorted(f for f in os.listdir(carpeta) if f.lower().endswith(EXT))

def hora(nombre):
    m = re.search(r"(\d{8}_\d{6})", nombre)
    return datetime.strptime(m.group(1), "%Y%m%d_%H%M%S") if m else None

def agrupar_por_hora(fotos):
    fotos = sorted(fotos, key=lambda f: (hora(f) or datetime.min, f))
    grupos, prev = [], None
    for f in fotos:
        t = hora(f)
        if prev is None or t is None or (t - prev).total_seconds() > GAP_SEG:
            grupos.append([])
        grupos[-1].append(f)
        prev = t
    return grupos

V5N_RAW = f"{RUNS}/v5n/weights/best.pt"             # v5 sin ajuste: punto de partida del entrenamiento
for ruta in (MODELO_V4, MODELO_V5, V5N_RAW, ZIP_V1):
    assert os.path.exists(ruta), f"No encuentro {ruta}"
print("Configuracion lista")

## 3. Pasar `ordenadas_camara` a su lugar (si tomaste fotos nuevas)

In [ ]:
ORDENADAS = f"{DRIVE_DIR}/ordenadas_camara"
movidas = 0
if os.path.isdir(ORDENADAS):
    for corto, largo in NOMBRES.items():
        origen = f"{ORDENADAS}/{largo}"
        fotos = imagenes(origen)
        if not fotos:
            continue
        grupos = agrupar_por_hora(fotos)
        random.shuffle(grupos)
        n_test = max(1, round(len(grupos) * 0.15))
        for i, g in enumerate(grupos):
            destino = f"{DRIVE_DIR}/test_camara/{corto}" if i < n_test else f"{DRIVE_DIR}/{corto}"
            os.makedirs(destino, exist_ok=True)
            for f in g:
                shutil.move(f"{origen}/{f}", f"{destino}/cam_{f}")
                movidas += 1
print("Fotos movidas:", movidas)

## 4. Partición vieja (congelada) y `test_camara`

In [ ]:
shutil.rmtree(DATASET, ignore_errors=True)
zipfile.ZipFile(ZIP_V1).extractall(DATASET)
for split in ("train", "val", "test"):
    for corto, largo in NOMBRES.items():
        if os.path.isdir(f"{DATASET}/{split}/{corto}"):
            os.rename(f"{DATASET}/{split}/{corto}", f"{DATASET}/{split}/{largo}")

shutil.rmtree(CAMARA, ignore_errors=True)
for corto, largo in NOMBRES.items():
    os.makedirs(f"{CAMARA}/{largo}", exist_ok=True)
    for f in imagenes(f"{DRIVE_DIR}/test_camara/{corto}"):
        shutil.copy2(f"{DRIVE_DIR}/test_camara/{corto}/{f}", f"{CAMARA}/{largo}/{f}")

def conteo(raiz, splits=("train", "val", "test")):
    print(f"{'clase':<20}" + "".join(f"{s:>8}" for s in splits))
    for c in NOMBRES.values():
        print(f"{c:<20}" + "".join(f"{len(imagenes(f'{raiz}/{s}/{c}')):>8}" for s in splits))

conteo(DATASET)
print("test_camara:", {c: len(imagenes(f"{CAMARA}/{c}")) for c in NOMBRES.values()})

## 5. Examen del docente (opcional, solo para MEDIR)
Sube a `data basurita/` el archivo **`examen_test.zip`** (solo las carpetas `test`), o las carpetas sueltas en `data basurita/examen/`:
```
examen/garbage.v1i.folder/test/{cardboard,glass,metal,paper,plastic}/
examen/Garbage Classification.v4i.folder/test/{Anorganik,Organik,Residu}/
examen/Smart medical waste segregation.v3i.folder/test/{General,Infectious}/
```
Solo se usa la carpeta `test`. **Nunca se entrena con ella**: además, cualquier foto de entrenamiento casi igual a una del examen se descarta.

In [ ]:
# grupo del docente -> colores aceptados (su protocolo exacto)
GRUPOS_EXAMEN = {
    "cardboard":  ("garbage", "Azul (papel/cartón)", {"Azul"}, True),
    "paper":      ("garbage", "Azul (papel/cartón)", {"Azul"}, True),
    "metal":      ("garbage", "Amarillo (metal)", {"Amarillo"}, True),
    "plastic":    ("garbage", "Blanco (plástico)", {"Blanco"}, True),
    "glass":      ("garbage", "Gris (vidrio)", {"Gris"}, True),
    "organik":    ("garbage_v4", "Marrón (Organik, grises)", {"Marron"}, True),
    "anorganik":  ("garbage_v4", "Inorg. reciclable (Anorganik)", {"Amarillo", "Azul", "Blanco", "Gris"}, False),
    "residu":     ("garbage_v4", "Negro/Rojo (Residu)", {"Negro", "Rojo"}, False),
    "infectious": ("medico", "Rojo (biocontaminado)", {"Rojo"}, True),
    "general":    ("medico", "No-Rojo (médico general)", {"Amarillo", "Azul", "Blanco", "Gris", "Marron", "Negro"}, False),
}

# Si subiste el examen como un solo zip (examen_test.zip) a 'data basurita', se descomprime aqui
EXAMEN_ZIP = f"{DRIVE_DIR}/examen_test.zip"
if os.path.exists(EXAMEN_ZIP):
    shutil.rmtree("/content/examen", ignore_errors=True)
    zipfile.ZipFile(EXAMEN_ZIP).extractall("/content/examen")
    EXAMEN_DIR = "/content/examen"
    print("Examen descomprimido desde", EXAMEN_ZIP)

EXAMEN = []   # (ruta, dataset, grupo, colores_ok, estricta, clase)
if os.path.isdir(EXAMEN_DIR):
    for raiz, dirs, archivos in os.walk(EXAMEN_DIR):
        partes = raiz.replace("\\", "/").split("/")
        if "test" not in partes:
            continue
        clase = os.path.basename(raiz).lower()
        if clase not in GRUPOS_EXAMEN:
            continue
        ds, grupo, ok, estricta = GRUPOS_EXAMEN[clase]
        for f in archivos:
            if f.lower().endswith(EXT):
                EXAMEN.append((os.path.join(raiz, f), ds, grupo, ok, estricta, clase))
import pandas as pd
if EXAMEN:
    EXAMEN = pd.DataFrame(EXAMEN, columns=["ruta", "dataset", "grupo", "colores_ok", "estricta", "clase"])
    print(f"Examen cargado: {len(EXAMEN)} imagenes (el docente uso 1455)")
    print(EXAMEN.groupby("grupo").size())
else:
    EXAMEN = None
    print("No hay examen en", EXAMEN_DIR, "-> se omite la nota estimada")

# Para NO entrenar con fotos del examen: nombre original (antes de que Roboflow lo renombrara) y hora de captura
import numpy as np
def tallo(nombre):
    b = os.path.basename(nombre)
    m = re.match(r"(.+?)_(jpg|jpeg|png)\.rf\.", b, re.I)
    b = m.group(1) if m else os.path.splitext(b)[0]
    return re.sub(r"[^a-z0-9]", "", b.lower())

def marca_ms(nombre):                      # Medical Waste 4.0 nombra sus fotos con la hora en milisegundos
    m = re.search(r"(?<!\d)(1[5-7]\d{11})(?!\d)", os.path.basename(nombre))
    return int(m.group(1)) if m else None

if EXAMEN is not None:
    TALLOS_EXAMEN = {tallo(r) for r in EXAMEN.ruta}
    TIEMPOS_EXAMEN = np.array(sorted({t for t in map(marca_ms, EXAMEN.ruta) if t}), dtype=np.int64)
    print(f"Filtro anti-examen: {len(TALLOS_EXAMEN)} nombres originales | {len(TIEMPOS_EXAMEN)} horas de captura (Medical Waste 4.0)")
else:
    TALLOS_EXAMEN, TIEMPOS_EXAMEN = set(), np.array([], dtype=np.int64)
    USAR_TRASHNET = USAR_WASTE_CLS_TECHSASH = USAR_MEDICAL_WASTE_40 = False
    print("AVISO: sin examen no se puede filtrar -> TrashNet, techsash y Medical Waste 4.0 se DESACTIVAN")

def es_del_examen(ruta):
    if tallo(ruta) in TALLOS_EXAMEN:
        return True
    t = marca_ms(ruta)
    return t is not None and len(TIEMPOS_EXAMEN) > 0 and int(np.abs(TIEMPOS_EXAMEN - t).min()) <= SESION_MW_SEG * 1000

## 6. Descargar datasets públicos
Se imprime qué categoría fue a qué color: **revísalo**.

In [ ]:
import kagglehub, requests
from PIL import Image, ImageOps
from concurrent.futures import ThreadPoolExecutor

publicos = defaultdict(lambda: defaultdict(list))   # color -> categoria -> [rutas]
asignacion = Counter()

def registrar(fuente, cat, color, rutas):
    asignacion[(fuente, cat, color or "ignorada")] += len(rutas)
    if color:
        publicos[color][f"{fuente}/{cat}"] += rutas

def fotos_en(raiz):
    return sorted(os.path.join(raiz, f) for f in os.listdir(raiz) if f.lower().endswith(EXT))

dominio = defaultdict(lambda: defaultdict(list))    # fuentes del mismo tipo que el examen (sin sus fotos)
quitadas_examen = Counter()
def registrar_dominio(fuente, cat, color, rutas):
    limpias = [r for r in rutas if not es_del_examen(r)]
    quitadas_examen[fuente] += len(rutas) - len(limpias)
    asignacion[(fuente, cat, color)] += len(limpias)
    dominio[color][f"{fuente}/{cat}"] += limpias

# --- 6.1 Recyclable and Household Waste (Kaggle, solo real_world) ---
REGLAS_HH = [("aerosol", "Rojo"), ("glass", "Gris"), ("styrofoam", "Negro"), ("paper_cup", "Negro"),
             ("_cans", "Amarillo"), ("cardboard", "Azul"), ("newspaper", "Azul"), ("magazine", "Azul"),
             ("office_paper", "Azul"), ("food_waste", "Marron"), ("coffee", "Marron"), ("eggshell", "Marron"),
             ("tea_bag", "Marron"), ("plastic_water", "Blanco"), ("plastic_soda", "Blanco"),
             ("plastic_detergent", "Blanco"), ("plastic_food_container", "Blanco"),
             ("plastic_cup_lid", "Blanco"), ("cutlery", "Blanco"), ("plastic_straws", "Blanco"),
             ("plastic_shopping_bags", "Blanco"), ("plastic_trash_bags", "Blanco")]
def color_por_reglas(nombre, reglas):
    for clave, color in reglas:
        if clave in nombre.lower():
            return color
    return None

RUTA_HH = kagglehub.dataset_download("alistairking/recyclable-and-household-waste-classification")
for raiz, _, archivos in os.walk(RUTA_HH):
    if os.path.basename(raiz).lower() == "real_world" and any(f.lower().endswith(EXT) for f in archivos):
        cat = os.path.basename(os.path.dirname(raiz))
        registrar("hh", cat, color_por_reglas(cat, REGLAS_HH), fotos_en(raiz))

# --- 6.2 RealWaste (UCI) ---
RW_DIR = "/content/realwaste"
if not os.path.isdir(RW_DIR):
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/908/realwaste.zip", "/content/realwaste.zip")
    zipfile.ZipFile("/content/realwaste.zip").extractall(RW_DIR)
    for z in glob.glob(f"{RW_DIR}/**/*.zip", recursive=True):
        zipfile.ZipFile(z).extractall(os.path.dirname(z))
MAPEO_RW = {"cardboard": "Azul", "paper": "Azul", "food organics": "Marron", "vegetation": "Marron",
            "glass": "Gris", "metal": "Amarillo", "plastic": "Blanco", "miscellaneous trash": "Negro"}
for raiz, _, archivos in os.walk(RW_DIR):
    if any(f.lower().endswith(EXT) for f in archivos):
        cat = os.path.basename(raiz)
        registrar("realwaste", cat, MAPEO_RW.get(cat.strip().lower()), fotos_en(raiz))

# --- 6.3 Medical Waste 4.0 (Zenodo) -> Rojo: RGB y estereo en grises (el examen usa los dos) ---
MW_DIR = "/content/medical_waste"
try:
    if not USAR_MEDICAL_WASTE_40:
        raise RuntimeError("desactivado")
    if not os.path.isdir(MW_DIR):
        print("Descargando Medical Waste 4.0 (1-2 GB, unos minutos)...")
        urllib.request.urlretrieve("https://zenodo.org/records/7643417/files/Medical%20Waste%204.0.zip?download=1",
                                   "/content/mw.zip")
        zipfile.ZipFile("/content/mw.zip").extractall(MW_DIR)
        os.remove("/content/mw.zip")
    TIPOS_MW = ("gauze", "glove", "cap", "glasses", "shoe", "tube", "urine")
    por_tipo = defaultdict(list)
    for raiz, _, archivos in os.walk(MW_DIR):
        if "__MACOSX" in raiz:
            continue
        for f in archivos:
            if f.lower().endswith(EXT):
                tipo = next((t for t in TIPOS_MW if t in raiz.lower()), "otros")
                vista = "estereo" if re.search(r"_(l|r)\.", f.lower()) else "rgb"
                por_tipo[f"{tipo}_{vista}"].append(os.path.join(raiz, f))
    for tipo, rutas in por_tipo.items():
        registrar_dominio("medical4.0", tipo, "Rojo", rutas)
except Exception as e:
    print("AVISO: Medical Waste 4.0 no se usa ->", e)

# --- 6.4 Waste Classification Data (Kaggle techsash) -> clase O = Marron ---
try:
    if not USAR_WASTE_CLS_TECHSASH:
        raise RuntimeError("desactivado")
    RUTA_WC = kagglehub.dataset_download("techsash/waste-classification-data")
    vistos, rutas_o = set(), []
    for raiz, _, archivos in os.walk(RUTA_WC):
        if os.path.basename(raiz).upper() == "O":
            for f in archivos:
                if f.lower().endswith(EXT) and f not in vistos:      # el zip trae carpetas repetidas
                    vistos.add(f); rutas_o.append(os.path.join(raiz, f))
    registrar_dominio("techsash", "O", "Marron", rutas_o)
except Exception as e:
    print("AVISO: Waste Classification no se usa ->", e)

# --- 6.4b TrashNet completo (Hugging Face) -> carton, papel, metal, plastico, vidrio, basura ---
TN_DIR = "/content/trashnet"
try:
    if not USAR_TRASHNET:
        raise RuntimeError("desactivado")
    if not os.path.isdir(TN_DIR):
        urllib.request.urlretrieve("https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip",
                                   "/content/trashnet.zip")
        zipfile.ZipFile("/content/trashnet.zip").extractall(TN_DIR)
    MAPEO_TN = {"cardboard": "Azul", "paper": "Azul", "metal": "Amarillo", "plastic": "Blanco",
                "glass": "Gris", "trash": "Negro"}
    for raiz, _, archivos in os.walk(TN_DIR):
        cat = os.path.basename(raiz).lower()
        if "__MACOSX" not in raiz and cat in MAPEO_TN:
            registrar_dominio("trashnet", cat, MAPEO_TN[cat], fotos_en(raiz))
except Exception as e:
    print("AVISO: TrashNet no se usa ->", e)

# --- 6.5 TACO: recortes de objetos (envolturas, colillas, tecnopor, comida, pilas, aerosoles) ---
REGLAS_TACO = [("crisp packet", "Negro"), ("other plastic wrapper", "Negro"), ("cigarette", "Negro"),
               ("styrofoam", "Negro"), ("tissues", "Negro"), ("food waste", "Marron"),
               ("battery", "Rojo"), ("aerosol", "Rojo"), ("plastic straw", "Blanco"), ("paper cup", "Negro")]
TACO_DIR = "/content/taco"
try:
    os.makedirs(f"{TACO_DIR}/img", exist_ok=True)
    ann = json.load(urllib.request.urlopen(
        "https://raw.githubusercontent.com/pedropro/TACO/master/data/annotations.json"))
    cat_color = {c["id"]: (c["name"], color_por_reglas(c["name"], REGLAS_TACO)) for c in ann["categories"]}
    info_img = {i["id"]: i for i in ann["images"]}
    por_img = defaultdict(list)
    for a in ann["annotations"]:
        if cat_color[a["category_id"]][1]:
            por_img[a["image_id"]].append(a)

    def bajar(iid):
        destino = f"{TACO_DIR}/img/{iid}.jpg"
        if os.path.exists(destino):
            return destino
        try:
            r = requests.get(info_img[iid]["flickr_url"], timeout=40)
            if r.ok:
                open(destino, "wb").write(r.content)
                return destino
        except Exception:
            pass
        return None

    with ThreadPoolExecutor(16) as ex:
        bajadas = dict(zip(por_img, ex.map(bajar, list(por_img))))
    recortes = defaultdict(list)
    for iid, anns in por_img.items():
        ruta = bajadas.get(iid)
        if not ruta:
            continue
        try:
            im = Image.open(ruta)
            w, h = info_img[iid]["width"], info_img[iid]["height"]
            if im.size != (w, h):
                im = ImageOps.exif_transpose(im)
            if im.size != (w, h):
                continue
            im = im.convert("RGB")
        except Exception:
            continue
        for a in anns:
            x, y, bw, bh = a["bbox"]
            mx, my = bw * 0.15, bh * 0.15
            caja = (max(0, x - mx), max(0, y - my), min(w, x + bw + mx), min(h, y + bh + my))
            rec = im.crop(tuple(int(v) for v in caja))
            if min(rec.size) < 40:
                continue
            escala = 160 / min(rec.size)
            if escala > 1:
                rec = rec.resize((int(rec.width * escala), int(rec.height * escala)))
            nombre, color = cat_color[a["category_id"]]
            os.makedirs(f"{TACO_DIR}/crops/{color}", exist_ok=True)
            destino = f"{TACO_DIR}/crops/{color}/taco_{a['id']}.jpg"
            rec.save(destino, quality=92)
            recortes[(nombre, color)].append(destino)
    for (nombre, color), rutas in recortes.items():
        registrar("taco", nombre, color, rutas)
    print(f"TACO: {sum(v is not None for v in bajadas.values())}/{len(bajadas)} fotos bajadas de Flickr")
except Exception as e:
    print("AVISO: TACO no disponible ->", e)

# --- 6.6 EPP medico publico -> Rojo (sin las clases de Medical Waste 4.0) ---
RUTAS_EPP = set()
try:
    if not USAR_EPP_PUBLICO:
        raise RuntimeError("desactivado")
    RUTA_MED = kagglehub.dataset_download("sumn2u/medical-waste")
    QUEDAR = ("mask", "syringe", "needle", "tweezer", "scalpel", "ampoule")
    EXCLUIR = ("glove", "gauze", "cap", "glasses", "shoe", "tube", "urine", "tissue", "organ", "packag", "organic")
    por_tipo = defaultdict(list)
    for raiz, _, archivos in os.walk(RUTA_MED):
        nombre_dir = os.path.basename(raiz).lower()
        if any(x in nombre_dir for x in EXCLUIR) or not any(x in nombre_dir for x in QUEDAR):
            continue
        for f in archivos:
            # fuera fotos con formato de Medical Waste 4.0 (marca de tiempo de 13 digitos, pares _l/_r)
            if f.lower().endswith(EXT) and not re.match(r"\d{13}", f) and not re.search(r"_(l|r)\.", f.lower()):
                por_tipo[nombre_dir].append(os.path.join(raiz, f))
    for tipo, rutas in por_tipo.items():
        registrar("epp", tipo, "Rojo", rutas)
        RUTAS_EPP.update(rutas)
except Exception as e:
    print("EPP publico no se usa ->", e)

# --- 6.7 Mas EPP biomedico publico -> Rojo (mismo filtro que 6.6) ---
try:
    if not USAR_BIOMEDICO_EXTRA:
        raise RuntimeError("desactivado")
    RUTA_BIO = kagglehub.dataset_download("mario78/biomedical-waste-classification-images-dataset")
    QUEDAR_B = ("syringe", "ampoule", "ampul", "scalpel", "needle", "mask", "blade", "vial", "lancet")
    EXCLUIR_B = ("glove", "gauze", "cap", "glasses", "shoe", "tube", "urine")
    por_tipo = defaultdict(list)
    for raiz, _, archivos in os.walk(RUTA_BIO):
        nombre_dir = os.path.basename(raiz).lower()
        if any(x in nombre_dir for x in EXCLUIR_B) or not any(x in nombre_dir for x in QUEDAR_B):
            continue
        for f in archivos:
            if f.lower().endswith(EXT) and not re.match(r"\d{13}", f) and not re.search(r"_(l|r)\.", f.lower()):
                por_tipo[nombre_dir].append(os.path.join(raiz, f))
    for tipo, rutas in por_tipo.items():
        registrar("bio", tipo, "Rojo", rutas)
        RUTAS_EPP.update(rutas)
except Exception as e:
    print("Biomedico extra no se usa ->", e)

# --- 6.8 Frutas podridas -> Marron ---
try:
    if not USAR_FRUTAS_PODRIDAS:
        raise RuntimeError("desactivado")
    RUTA_FR = kagglehub.dataset_download("sriramr/fruits-fresh-and-rotten-for-classification")
    por_tipo = defaultdict(list)
    for raiz, _, archivos in os.walk(RUTA_FR):
        nombre_dir = os.path.basename(raiz).lower()
        if "rotten" in nombre_dir:
            por_tipo[nombre_dir] += [os.path.join(raiz, f) for f in archivos if f.lower().endswith(EXT)]
    for tipo, rutas in por_tipo.items():
        registrar("frutas", tipo, "Marron", rutas)
except Exception as e:
    print("Frutas podridas no se usa ->", e)

print(f"\n{'fuente':<12}{'categoria':<42}{'color':<10}{'fotos':>6}")
for (fuente, cat, color), n in sorted(asignacion.items()):
    print(f"{fuente:<12}{cat[:41]:<42}{color:<10}{n:>6}")

def elegir_balanceado(por_cat, tope):
    # mismo resultado en cualquier sesion de Colab: orden fijo y una semilla por categoria
    grupos = []
    for cat in sorted(por_cat):
        l = sorted(set(por_cat[cat]))
        random.Random(f"{SEMILLA}|{cat}").shuffle(l)
        grupos.append(l)
    elegidas = []
    while len(elegidas) < tope and any(grupos):
        for g in grupos:
            if g and len(elegidas) < tope:
                elegidas.append(g.pop())
    return elegidas

PUBLICO_ELEGIDO = {c: elegir_balanceado(publicos[c], MAX_PUBLICO_POR_COLOR) for c in NOMBRES}
DOMINIO_ELEGIDO = {c: elegir_balanceado(dominio[c], MAX_DOMINIO_POR_COLOR) for c in NOMBRES}
print("\nCandidatas publicas por color:", {c: len(v) for c, v in PUBLICO_ELEGIDO.items()})
print("Candidatas del mismo tipo que el examen:", {c: len(v) for c, v in DOMINIO_ELEGIDO.items()})
print("Quitadas por ser del examen (nombre o sesion):", dict(quitadas_examen))

## 7. ETL – limpieza
Se descartan: ilegibles, miniaturas, borrosas, las que ya estaban en v1 y las **casi iguales a cualquier test**
(test viejo, `test_camara` y **examen**). Luego el tope de fotos de celular por color.

In [ ]:
import numpy as np, imagehash, cv2
import pandas as pd

MIN_LADO, MIN_KB, BLUR_MIN = 128, 5, 20

def huella(ruta):
    try:
        with Image.open(ruta) as im:
            return int(str(imagehash.phash(ImageOps.exif_transpose(im).convert("RGB"))), 16)
    except Exception:
        return None

def inspeccionar(ruta):
    try:
        with Image.open(ruta) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            w, h = im.size
            h_p = str(imagehash.phash(im))
            im.thumbnail((LADO_MAX, LADO_MAX))
            nitidez = float(cv2.Laplacian(cv2.cvtColor(np.array(im), cv2.COLOR_RGB2GRAY), cv2.CV_64F).var())
        return {"ancho": w, "alto": h, "kb": os.path.getsize(ruta) / 1024, "nitidez": nitidez, "huella": h_p}
    except Exception:
        return None

def huellas(rutas):
    hs = [huella(r) for r in rutas]
    return np.array([h for h in hs if h is not None], dtype=np.uint64)

def distancia_min(h, arr):
    if len(arr) == 0:
        return 64
    x = np.bitwise_xor(arr, np.uint64(h))
    return int(np.unpackbits(x.view(np.uint8)).reshape(-1, 64).sum(1).min())

viejas = [f"{DATASET}/{s}/{c}/{f}" for s in ("train", "val", "test") for c in NOMBRES.values()
          for f in imagenes(f"{DATASET}/{s}/{c}")]
de_test = [f"{DATASET}/test/{c}/{f}" for c in NOMBRES.values() for f in imagenes(f"{DATASET}/test/{c}")]
de_test += [f"{CAMARA}/{c}/{f}" for c in NOMBRES.values() for f in imagenes(f"{CAMARA}/{c}")]
if EXAMEN is not None:
    de_test += list(EXAMEN.ruta)
print("Calculando huellas (unos minutos)...")
H_VIEJAS, H_TEST = huellas(viejas), huellas(de_test)
print(f"viejas: {len(H_VIEJAS)} | tests a proteger: {len(H_TEST)}")

candidatas = []
for corto, largo in NOMBRES.items():
    for f in imagenes(f"{DRIVE_DIR}/{corto}"):
        candidatas.append((f"{DRIVE_DIR}/{corto}/{f}", largo, "cam" if f.startswith("cam_") else "web"))
    for ruta in PUBLICO_ELEGIDO[corto]:
        candidatas.append((ruta, largo, "publico"))
    for ruta in DOMINIO_ELEGIDO[corto]:
        candidatas.append((ruta, largo, "dominio"))

registros = []
for ruta, largo, fuente in candidatas:
    info = inspeccionar(ruta)
    if info is None:
        registros.append({"ruta": ruta, "clase": largo, "fuente": fuente, "estado": "ilegible"}); continue
    hh = int(info["huella"], 16)
    if min(info["ancho"], info["alto"]) < MIN_LADO or (fuente in ("cam", "web") and info["kb"] < MIN_KB):
        estado = "pequena"
    elif distancia_min(hh, H_VIEJAS) <= 4:
        estado = "ya_en_v1"
    elif distancia_min(hh, H_TEST) <= (12 if (ruta in RUTAS_EPP or fuente == "dominio") else 8):
        estado = "parecida_a_test"
    elif info["nitidez"] < BLUR_MIN:
        estado = "borrosa"
    else:
        estado = "nueva"
    registros.append({"ruta": ruta, "clase": largo, "fuente": fuente, "estado": estado, **info})
ETL = pd.DataFrame(registros)

for largo in NOMBRES.values():
    sel = ETL[(ETL.clase == largo) & (ETL.fuente == "cam") & (ETL.estado == "nueva")]
    if len(sel) <= MAX_CAM_POR_COLOR:
        continue
    por_nombre = {os.path.basename(r): i for i, r in zip(sel.index, sel.ruta)}
    rng_cam = random.Random(f"{SEMILLA}|{largo}")          # mismo resultado en cualquier sesion
    grupos = [rng_cam.sample(g, len(g)) for g in agrupar_por_hora(sorted(por_nombre))]
    elegidas = set()
    while len(elegidas) < MAX_CAM_POR_COLOR:
        for g in grupos:
            if g and len(elegidas) < MAX_CAM_POR_COLOR:
                elegidas.add(por_nombre[g.pop()])
    ETL.loc[sel.index.difference(list(elegidas)), "estado"] = "sobra_tope_cam"

print(pd.crosstab([ETL.clase, ETL.fuente], ETL.estado, margins=True, margins_name="TOTAL"))
nuevas = defaultdict(list)
for r in ETL[ETL.estado == "nueva"].itertuples():
    nuevas[r.clase].append((r.ruta, int(r.huella, 16), r.fuente))

## 8. Train/val de lo nuevo (por objeto) + **40% del train en escala de grises**
El docente baja puntos si el modelo depende del color (3 de sus 9 grupos están en grises).

In [ ]:
import hashlib

def grupos_por_huella(items):
    padre = list(range(len(items)))
    def raiz(i):
        while padre[i] != i:
            padre[i] = padre[padre[i]]; i = padre[i]
        return i
    hs = np.array([h for _, h, _ in items], dtype=np.uint64)
    for i in range(len(items)):
        x = np.bitwise_xor(hs[i + 1:], hs[i])
        if len(x):
            d = np.unpackbits(x.view(np.uint8)).reshape(-1, 64).sum(1)
            for j in np.nonzero(d <= 6)[0]:
                padre[raiz(i + 1 + j)] = raiz(i)
    g = defaultdict(list)
    for i, it in enumerate(items):
        g[raiz(i)].append(it)
    return list(g.values())

def guardar(origen, destino):
    with Image.open(origen) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        im.thumbnail((LADO_MAX, LADO_MAX))
        im.save(destino, "JPEG", quality=90)

agregadas = defaultdict(Counter)
for largo, items in nuevas.items():
    cam = [it for it in items if it[2] == "cam"]
    otras = [it for it in items if it[2] != "cam"]
    por_nombre = {os.path.basename(r): (r, h, s) for r, h, s in cam}
    # Medical Waste 4.0: las fotos de la misma captura (RGB + par estereo, tomadas a <5 s) van juntas
    con_hora = sorted((it for it in otras if marca_ms(it[0])), key=lambda it: marca_ms(it[0]))
    grupos_mw, prev = [], None
    for it in con_hora:
        t = marca_ms(it[0])
        if prev is None or t - prev > 5000:
            grupos_mw.append([])
        grupos_mw[-1].append(it); prev = t
    sin_hora = [it for it in otras if not marca_ms(it[0])]
    grupos = [[por_nombre[f] for f in g] for g in agrupar_por_hora(list(por_nombre))] + grupos_mw + grupos_por_huella(sin_hora)
    # train/val decidido por un hash del objeto: la MISMA foto cae SIEMPRE en el mismo lado, aunque Colab se reinicie
    for g in grupos:
        clave = min(os.path.basename(r) for r, _, _ in g)
        split = "val" if int(hashlib.md5(f"{SEMILLA}|{largo}|{clave}".encode()).hexdigest(), 16) % 1000 < PCT_VAL * 1000 else "train"
        os.makedirs(f"{DATASET}/{split}/{largo}", exist_ok=True)
        for ruta, _, fuente in g:
            base = re.sub(r"[^\w.-]", "_", os.path.splitext(os.path.basename(ruta))[0])[:50]
            uid = hashlib.md5("/".join(ruta.replace("\\", "/").split("/")[-4:]).encode()).hexdigest()[:10]
            try:
                guardar(ruta, f"{DATASET}/{split}/{largo}/v6_{fuente}_{uid}_{base}.jpg")
                agregadas[largo][f"{split}_{fuente}"] += 1
            except Exception as e:
                print("omitida:", ruta, e)
print(pd.DataFrame(agregadas).T.fillna(0).astype(int))

# Escala de grises en una parte del TRAIN (en el lugar, sin duplicar)
grises = 0
for c in NOMBRES.values():
    fotos = imagenes(f"{DATASET}/train/{c}")
    for f in random.sample(fotos, int(len(fotos) * PCT_GRIS)):
        ruta = f"{DATASET}/train/{c}/{f}"
        Image.open(ruta).convert("L").convert("RGB").save(ruta, "JPEG", quality=92)
        grises += 1
print(f"\nFotos de train pasadas a grises: {grises}")

# Copia en grises del val, para elegir el modelo tambien por robustez al color
VAL_GRIS = "/content/val_gris"
shutil.rmtree(VAL_GRIS, ignore_errors=True)
for c in NOMBRES.values():
    os.makedirs(f"{VAL_GRIS}/{c}", exist_ok=True)
    for f in imagenes(f"{DATASET}/val/{c}"):
        Image.open(f"{DATASET}/val/{c}/{f}").convert("L").convert("RGB").save(f"{VAL_GRIS}/{c}/{f}")
conteo(DATASET)

## 9. Pre-EDA

In [ ]:
import matplotlib.pyplot as plt

def fuente_de(nombre):
    for pref, etiqueta in (("v6_cam", "celular"), ("v6_web", "internet"), ("v6_publico", "publico"),
                           ("v6_dominio", "mismo tipo que examen")):
        if nombre.startswith(pref):
            return etiqueta
    return "v1 (original)"

filas = [{"split": s, "clase": c, "fuente": fuente_de(f)}
         for s in ("train", "val", "test") for c in NOMBRES.values() for f in imagenes(f"{DATASET}/{s}/{c}")]
INVENTARIO = pd.DataFrame(filas)
tr = pd.crosstab(INVENTARIO[INVENTARIO.split == "train"].clase, INVENTARIO[INVENTARIO.split == "train"].fuente)
print(tr)
print(f"Desbalance en train: {tr.sum(axis=1).max() / tr.sum(axis=1).min():.2f}x")
tr.plot.barh(stacked=True, figsize=(9, 4), title="Train por color y fuente"); plt.tight_layout(); plt.show()

def galeria(rutas, titulos, titulo, n=12):
    rutas, titulos = list(rutas)[:n], list(titulos)[:n]
    if not rutas:
        print(titulo, "-> (nada)"); return
    cols = 6; filas_g = (len(rutas) + cols - 1) // cols
    fig, ejes = plt.subplots(filas_g, cols, figsize=(2.3 * cols, 2.5 * filas_g), squeeze=False)
    for ax in ejes.flat: ax.axis("off")
    for ax, r, t in zip(ejes.flat, rutas, titulos):
        with Image.open(r) as im:
            ax.imshow(ImageOps.exif_transpose(im).convert("RGB"))
        ax.set_title(t, fontsize=8)
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

nuevas_pub = ETL[(ETL.estado == "nueva") & (ETL.fuente == "dominio")]
for color in ("Rojo", "Marron", "Blanco", "Gris"):
    m = nuevas_pub[nuevas_pub.clase == NOMBRES[color]].sample(frac=1, random_state=1)
    galeria(m.ruta, [os.path.basename(os.path.dirname(r))[:14] for r in m.ruta],
            f"Fotos nuevas (mismo tipo que examen) en {color}: revisa que tengan sentido")

## 10. Entrenar v6n desde v5
Mismo esquema de siempre (SGD, aprendizaje bajo, early stopping, `best.pt` con val).
Si ya existe en Drive **y** se entrenó con exactamente la misma partición (se guarda en `v6n_particion.json`),
**no se vuelve a entrenar**. Así, si Colab se desconecta, val nunca contiene fotos que el modelo ya vio.

In [ ]:
from ultralytics import YOLO

V6N = f"{RUNS}/v6n/weights/best.pt"
MANIFIESTO = f"{RUNS}/v6n_particion.json"     # que fotos fueron a train y a val cuando se entreno
ACTUAL = {s: sorted(f"{c}/{f}" for c in NOMBRES.values() for f in imagenes(f"{DATASET}/{s}/{c}") if f.startswith("v6_"))
          for s in ("train", "val")}

reusar = os.path.exists(V6N) and os.path.exists(MANIFIESTO)
if reusar:
    fuga = set(ACTUAL["val"]) & set(json.load(open(MANIFIESTO))["train"])
    if fuga:
        print(f"{len(fuga)} fotos de val estaban en el train del modelo guardado -> se reentrena (si no, val saldria inflado)")
        reusar = False
if reusar:
    print("v6n ya estaba entrenado con esta misma particion -> se reutiliza:", V6N)
else:
    if os.path.isdir(f"{RUNS}/v6n"):
        destino = f"{RUNS}/v6n_descartado_{datetime.now():%Y%m%d_%H%M}"
        shutil.move(f"{RUNS}/v6n", destino)
        print("El entrenamiento anterior no se puede verificar contra esta particion -> se guarda aparte en", destino)
    json.dump(ACTUAL, open(MANIFIESTO, "w"))
    YOLO(V5N_RAW).train(data=DATASET, epochs=30, imgsz=224, batch=64, optimizer="SGD", lr0=0.001,
                        lrf=0.1, warmup_epochs=0, patience=6, seed=SEMILLA, project=RUNS,
                        name="v6n", exist_ok=True)
CANDIDATOS = {"v5": MODELO_V5, "v6n": V6N}
print(CANDIDATOS)

## 11. Ajuste fino de los 7 colores y elección **con val**
A cada color se le suma un pequeño valor a su puntaje; los 7 valores se eligen para **maximizar el acierto balanceado
en val** (a color y en grises). El ajuste se **graba dentro del `.pt`** (sesgo de la última capa), así que funciona igual
con `YOLO("clasificador.pt").predict()`, que es como evalúa el docente.

In [ ]:
import torch

def probabilidades(ruta_modelo, rutas):
    m = YOLO(ruta_modelo)
    orden = [LARGO_A_CORTO[m.names[i]] for i in range(len(m.names))]
    P = np.array([r.probs.data.cpu().numpy() for r in m.predict(list(rutas), imgsz=224, verbose=False, stream=True)])
    return P, orden

def rutas_de(carpeta):
    rutas, reales = [], []
    for c in sorted(os.listdir(carpeta)):
        for f in imagenes(f"{carpeta}/{c}"):
            rutas.append(f"{carpeta}/{c}/{f}"); reales.append(LARGO_A_CORTO[c])
    return rutas, np.array(reales)

def balanceado(reales, preds):
    return float(np.mean([(preds[reales == c] == c).mean() for c in np.unique(reales)]))

def con_sesgos(P, orden, b):
    return np.array(orden)[(P * np.exp(b)).argmax(1)]

def calibrar(Pc, Pg, orden):
    objetivo = lambda b: (balanceado(VAL_Y, con_sesgos(Pc, orden, b)) + balanceado(VALG_Y, con_sesgos(Pg, orden, b))) / 2
    b = np.zeros(len(orden)); inicial = mejor = objetivo(b)
    for _ in range(3):                        # 3 vueltas, un color a la vez
        cambio = False
        for k in range(len(orden)):
            for v in np.arange(-1, 2.01, 0.25):
                bb = b.copy(); bb[k] = v
                p = objetivo(bb)
                if p > mejor + 1e-4:
                    mejor, b, cambio = p, bb, True
        if not cambio:
            break
    return b, inicial, mejor

def hornear_sesgos(ruta, b, orden, salida):
    ck = torch.load(ruta, map_location="cpu", weights_only=False)
    m = (ck["ema"] if ck.get("ema") is not None else ck["model"]).float()
    assert [LARGO_A_CORTO[m.names[i]] for i in range(len(m.names))] == orden
    m.model[-1].linear.bias.data += torch.tensor(b, dtype=torch.float32)
    ck["model"], ck["ema"] = m.half(), None
    torch.save(ck, salida)
    return salida

VAL_R, VAL_Y = rutas_de(f"{DATASET}/val")
VALG_R, VALG_Y = rutas_de(VAL_GRIS)
puntaje, sesgos = {}, {}
for nombre, ruta in list(CANDIDATOS.items()):
    Pc, orden = probabilidades(ruta, VAL_R)
    Pg, _ = probabilidades(ruta, VALG_R)
    b, inicial, mejor = calibrar(Pc, Pg, orden)
    puntaje[nombre] = inicial
    print(f"{nombre:<5} val (color+gris) {inicial:.2%} -> con ajuste {mejor:.2%} | ajuste: " +
          (", ".join(f"{c} {v:+.2f}" for c, v in zip(orden, b) if v) or "ninguno"))
    if np.any(b != 0):
        cal = f"{nombre}_cal"
        sesgos[cal] = (b, orden, nombre)
        CANDIDATOS[cal] = hornear_sesgos(ruta, b, orden, f"/content/{cal}.pt")
        puntaje[cal] = mejor

# comprobacion: el .pt horneado predice igual que la simulacion
for cal, (b, orden, base) in sesgos.items():
    Pb, _ = probabilidades(CANDIDATOS[base], VAL_R[:200])
    Pc, orden_c = probabilidades(CANDIDATOS[cal], VAL_R[:200])
    print(f"{cal}: el .pt horneado coincide con la simulacion en "
          f"{np.mean(con_sesgos(Pb, orden, b) == np.array(orden_c)[Pc.argmax(1)]):.1%} de 200 fotos")

ELEGIDO = max(puntaje, key=puntaje.get)
print("\nPuntaje en val:", {k: f"{v:.2%}" for k, v in puntaje.items()})
print("Elegido por val:", ELEGIDO)

## 12. Comparar en test viejo y `test_camara`

In [ ]:
def evaluar_carpeta(ruta_modelo, carpeta):
    rutas, reales = rutas_de(carpeta)
    P, orden = probabilidades(ruta_modelo, rutas)
    df = pd.DataFrame({"ruta": rutas, "real": reales, "pred": np.array(orden)[P.argmax(1)], "conf": P.max(1)})
    df["ok"] = df.real == df.pred
    return df

def por_objeto(df):
    df = df.assign(nombre=df.ruta.map(os.path.basename))
    ok = []
    for clase, g in df.groupby("real"):
        for grupo in agrupar_por_hora(list(g.nombre)):
            ok.append(g[g.nombre.isin(grupo)].pred.mode()[0] == clase)
    return sum(ok), len(ok)

MODELOS_CMP = {"v4n": MODELO_V4, **CANDIDATOS}
R = {(v, p): evaluar_carpeta(ruta, carpeta) for v, ruta in MODELOS_CMP.items()
     for p, carpeta in (("test_viejo", f"{DATASET}/test"), ("test_camara", CAMARA))}
print(f"{'':<20}" + "".join(f"{v:>10}" for v in MODELOS_CMP))
for p in ("test_viejo", "test_camara"):
    print(f"{p:<20}" + "".join(f"{R[(v, p)].ok.mean():>10.1%}" for v in MODELOS_CMP))
print(f"{'camara (objetos)':<20}" + "".join(f"{'%d/%d' % por_objeto(R[(v, 'test_camara')]):>10}" for v in MODELOS_CMP))

## 13. Nota estimada con la fórmula del docente (solo se MIDE, no se usa para elegir)
`nota = 8·exact.balanceada + 4·macroF1 + 3·colores/7 + (3 − 0.05·caída_grises) + seguridad`,
con seguridad = `3·orgánicos + 3.11·F1_Rojo`. Se muestra también con seguridad **recortada a 2** (por si el docente la limita).

In [ ]:
from sklearn.metrics import f1_score

RANKING_DOCENTE = {"15 Pastor": 16.0, "01 Jhoel": 14.0, "21 Jhosef": 13.6, "08 Fernandez": 13.2,
                   "11 Jorge": 12.9, "02 Cabana": 12.5, "03 Fran": 12.3, "13 Kana": 12.0,
                   "18 Montes": 11.9, "17 Ivan": 11.7, "16 Ciro": 11.2}

def nota_docente(ruta_modelo):
    ex = EXAMEN.copy()
    P, orden = probabilidades(ruta_modelo, ex.ruta)
    ex["pred"] = np.array(orden)[P.argmax(1)]
    ex["ok"] = [p in oks for p, oks in zip(ex.pred, ex.colores_ok)]
    bal = ex.groupby("grupo").ok.mean().mean()
    est = ex[ex.estricta]
    y = est.colores_ok.map(lambda s: next(iter(s)))
    mf1 = f1_score(y, est.pred, labels=sorted(y.unique()), average="macro", zero_division=0)
    med = ex[ex.dataset == "medico"]
    rec = (med[med.clase == "infectious"].pred == "Rojo").mean()
    pr = med[med.pred == "Rojo"]
    prec = (pr.clase == "infectious").mean() if len(pr) else 0
    f1r = 2 * prec * rec / (prec + rec) if prec + rec else 0
    org = ex[ex.clase == "organik"].ok.mean()
    v1i = ex[ex.dataset == "garbage"]
    tmp = "/content/_v1i_gris"; shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    rut_g = []
    for i, r in enumerate(v1i.ruta):
        d = f"{tmp}/{i}.jpg"; Image.open(r).convert("L").convert("RGB").save(d); rut_g.append(d)
    Pg, og = probabilidades(ruta_modelo, rut_g)
    acc_g = np.mean([p in oks for p, oks in zip(np.array(og)[Pg.argmax(1)], v1i.colores_ok)])
    caida = (v1i.ok.mean() - acc_g) * 100
    seg = 3 * org + 28 / 9 * f1r
    base = 8 * bal + 4 * mf1 + 3.0 + (3 - 0.05 * caida)
    met = {"nota": base + seg, "nota_seg_max2": base + min(seg, 2), "exact_bal": bal, "macroF1": mf1,
           "organicos": org, "recall_rojo": rec, "prec_rojo": prec, "caida_gris_pp": caida}
    return met, ex

if EXAMEN is not None:
    NOTAS = {v: nota_docente(ruta) for v, ruta in MODELOS_CMP.items()}
    print(pd.DataFrame({v: NOTAS[v][0] for v in NOTAS}).round(3))
    print("\nAcierto por grupo del docente:")
    POR_GRUPO = pd.DataFrame({v: NOTAS[v][1].groupby("grupo").ok.mean() for v in NOTAS})
    POR_GRUPO[f"{ELEGIDO} - v5"] = POR_GRUPO[ELEGIDO] - POR_GRUPO["v5"]
    print(POR_GRUPO.round(3).to_string())
    print("\nRanking (nota sin recortar):")
    todos = {**RANKING_DOCENTE, **{f"TU {v}": NOTAS[v][0]["nota"] for v in NOTAS}}
    for i, (k, n) in enumerate(sorted(todos.items(), key=lambda x: -x[1]), 1):
        print(f"{i:>2}. {k:<14}{n:5.1f}" + ("   <--" if k.startswith("TU") else ""))
else:
    print("Sin examen -> no hay nota estimada")

## 14. Diagnóstico: a qué color se van los errores
Para el informe: matriz de confusión del elegido en el examen (por grupo) y en val, y galería de errores.

In [ ]:
import matplotlib.pyplot as plt

def galeria(rutas, titulos, titulo, n=12):
    rutas, titulos = list(rutas)[:n], list(titulos)[:n]
    if not rutas:
        print(titulo, "-> (nada)"); return
    cols = 6; filas_g = (len(rutas) + cols - 1) // cols
    fig, ejes = plt.subplots(filas_g, cols, figsize=(2.3 * cols, 2.5 * filas_g), squeeze=False)
    for ax in ejes.flat: ax.axis("off")
    for ax, r, t in zip(ejes.flat, rutas, titulos):
        with Image.open(r) as im:
            ax.imshow(ImageOps.exif_transpose(im).convert("RGB"))
        ax.set_title(t, fontsize=8)
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

if EXAMEN is not None:
    ex = NOTAS[ELEGIDO][1]
    conf = pd.crosstab(ex.grupo, ex.pred, normalize="index").round(2)
    print(f"{ELEGIDO} en el examen: a que color va cada grupo (filas suman 1)")
    print(conf.to_string())
    os.makedirs(REVISION, exist_ok=True)
    conf.to_csv(f"{REVISION}/confusion_examen_{ELEGIDO}.csv")
    ex.drop(columns=["colores_ok"]).to_csv(f"{REVISION}/predicciones_examen_{ELEGIDO}.csv", index=False)
    err = ex[~ex.ok].sample(min(18, (~ex.ok).sum()), random_state=1)
    galeria(err.ruta, [f"{g[:14]} -> {p}" for g, p in zip(err.grupo, err.pred)], f"Errores de {ELEGIDO} en el examen", n=18)

dv = evaluar_carpeta(CANDIDATOS[ELEGIDO], f"{DATASET}/val")
print(f"\n{ELEGIDO} en val (filas = real):")
print(pd.crosstab(dv.real, dv.pred, normalize="index").round(2).to_string())

## 15. Decidir y guardar
`clasificador.pt` se reemplaza solo si el elegido **no es v5** y, frente a v5: **ningún color baja más de 2 puntos en val**,
el test viejo no baja más de 2 puntos y en `test_camara` no pierde más de 1 objeto.

In [ ]:
import hashlib
val_c = {v: evaluar_carpeta(CANDIDATOS[v], f"{DATASET}/val") for v in dict.fromkeys(("v5", ELEGIDO))}
REC = pd.DataFrame({v: d.groupby("real").ok.mean() for v, d in val_c.items()})
print("Acierto por color en val:"); print(REC.round(3).to_string())
ok_colores = bool((REC[ELEGIDO] >= REC["v5"] - 0.02).all())
ok_viejo = R[(ELEGIDO, "test_viejo")].ok.mean() >= R[("v5", "test_viejo")].ok.mean() - 0.02
o_e, _ = por_objeto(R[(ELEGIDO, "test_camara")]); o_5, _ = por_objeto(R[("v5", "test_camara")])
ok_cam = o_e >= o_5 - 1
GANA = ELEGIDO != "v5" and ok_colores and ok_viejo and ok_cam
shutil.copy(CANDIDATOS[ELEGIDO], f"{MODELOS}/clasificador_v6.pt")
if GANA:
    shutil.copy(CANDIDATOS[ELEGIDO], f"{MODELOS}/clasificador.pt")
    print(f"\n{ELEGIDO} GANA -> modelos/clasificador.pt ahora es v6 (copia en clasificador_v6.pt)")
else:
    print(f"\n{ELEGIDO} no gana (colores ok={ok_colores}, test viejo ok={ok_viejo}, objetos {o_e} vs {o_5})"
          " -> clasificador.pt sigue siendo v5")
print("md5 de clasificador.pt:", hashlib.md5(open(f"{MODELOS}/clasificador.pt", "rb").read()).hexdigest())
os.makedirs(REVISION, exist_ok=True)
ETL.drop(columns=["huella"], errors="ignore").to_csv(f"{REVISION}/reporte_etl.csv", index=False)
pd.Series(puntaje).to_csv(f"{REVISION}/eleccion_por_val.csv")
pd.DataFrame({k: dict(zip(o, b)) for k, (b, o, _) in sesgos.items()}).to_csv(f"{REVISION}/ajuste_colores.csv")
REC.to_csv(f"{REVISION}/val_por_color.csv")
print("Evidencias en", REVISION)

## 16. Descargar

In [ ]:
from google.colab import files
files.download(f"{MODELOS}/clasificador.pt")